# Рисование карты дефектов с помощью модели ver_10

In [1]:
# Подключенине библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import Data_processing as dp

from matplotlib import ticker
from IPython.display import display
from tensorflow import keras
from tensorflow.keras.layers import Reshape, Input, Dense, Flatten, Conv2D, Dropout, Conv2DTranspose
from tensorflow.keras.layers import MaxPooling2D, UpSampling2D, concatenate, BatchNormalization

2023-12-30 04:30:40.054827: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-30 04:30:40.142587: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-30 04:30:40.142638: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-30 04:30:40.142683: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-30 04:30:40.153077: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-12-30 04:30:42.712620: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

### Константы для обработки данных

In [ ]:
# Пути к обработанным данным
# Путь к X части выборки с массивами в каждой ячейке
# Run1 - данные обучающей выборки
# Run2 - данные тестовой выборки
# Путь к Y части выборки

# paths for files with original data
RUN_1_PATH = {'x':'data/Prepared data/Run1/X_data_array_like.xlsx',
             'y':'data/Prepared data/Run1/Y_data(binary_classification).xlsx'}

RUN_2_PATH = {'x':'data/Prepared data/Run2/X_data_array_like.xlsx',
             'y':'data/Prepared data/Run2/Y_data(binary_classification).xlsx'}

# Размер кропа
PREP_image_size = 32
# Шаг кропа
PREP_crop_step = 32

# Детерминация случайных величин, отвечающих за выбор первоначальных весов и биасов
tf.compat.v1.set_random_seed(290)
tf.random.set_seed(290)

### Функции для обработки

In [ ]:
def extend_df_for_prediction_2(df, crop_size, crop_step):

    print('||||||||||||||||||')
    print('Df extending for better prediction')
    print('Original df size: ', df.shape)
    print('Crop windows height/width: ', crop_size)
    print('Crop windows step across rows and cols: ', crop_step)
        
    extend_dims = crop_size - 1

    df = pd.concat([df.iloc[:,-1 * extend_dims:],
                    df,
                    df.iloc[:,:extend_dims]],axis=1,ignore_index=True)
    
    df = pd.concat([df.iloc[extend_dims-1::-1,:], 
                    df,
                    df.iloc[-1:-extend_dims-1:-1,:]
                    ],axis=0,ignore_index=True)
        
    print('New df shape: ', df.shape)
    print('||||||||||||||||||\n')
    
    dp.reshape_df_for_future_crops(df,crop_size,crop_step)

    return df

### 3 подхода к предобработке

In [ ]:
# Преобразовать массив результатов работы модели размера (batch,1)
# в матрицу размера PREP_df_rows * PREP_df_cols (карту дефектов, построенную моделью)
# Размер выходной матрицы - PREP_df_rows * PREP_df_cols
# Так как начальное расширение датафрейма на PREP_image_size по 2 осям
# тут учитывается грубо, то размер выходной матрицы будет также
# PREP_df_rows * PREP_df_cols, а ячейки по конрутру будут размыты 
# и не точны, так как по ним фильтр прошел меньше раз чем по ячейкам 
# посреди матрицы

def test_reshape_2D_Y_numpy_to_2D_0(rows_count, cols_count, crop_size, step = -1):

    print('||||||||||||||||||')
    print('Y arr reshaping to 2D')
    print('Crop windows height/width: ', crop_size)
    print('Crop windows step across rows and cols: ', step)

    if step == -1:
        step = crop_size
    
    new_arr = np.zeros((rows_count, cols_count))
      
    for j in range(0,  cols_count - crop_size + 1, step):
        for i in range(0, rows_count - crop_size + 1, step):
            new_arr[i:i+crop_size,j:j+crop_size] += 1      
            
    print('New numpy shape: ', new_arr.shape)
    print('||||||||||||||||||\n')

    return new_arr

In [ ]:
# Преобразовать массив результатов работы модели размера (batch,1)
# в матрицу размера PREP_df_rows * PREP_df_cols (карту дефектов, построенную моделью)
# Размер выходной матрицы - PREP_df_rows * PREP_df_cols
# Так как начальное расширение датафрейма на PREP_image_size по 2 осям
# тут учитывается грубо, то размер выходной матрицы будет также
# PREP_df_rows * PREP_df_cols, а ячейки по конрутру будут размыты 
# и не точны, так как по ним фильтр прошел меньше раз чем по ячейкам 
# посреди матрицы

def test_reshape_2D_Y_numpy_to_2D_2(orig_rows_count, orig_cols_count,
                                    result_rows_count, result_cols_count, 
                                    crop_size, step = -1):

    print('||||||||||||||||||')
    print('Y arr reshaping to 2D')
    print('Crop windows height/width: ', crop_size)
    print('Crop windows step across rows and cols: ', step)

    if step == -1:
        step = crop_size
    
    new_arr = np.zeros((result_rows_count, result_cols_count))
      
    for j in range(0,  result_cols_count - crop_size + 1, step):
        for i in range(0, result_rows_count - crop_size + 1, step):
            new_arr[i:i+crop_size,j:j+crop_size] += 1      
    
    print('New numpy shape: ', new_arr.shape)
    print('||||||||||||||||||\n')

    left_up_border = crop_size - 1
    right_border = result_cols_count - (result_cols_count - orig_cols_count) + 1
    bottom_border = result_rows_count - (result_rows_count - orig_rows_count) + 1
    
    return new_arr[left_up_border:,left_up_border:][:orig_rows_count,:orig_cols_count]

In [ ]:
test_rows = dp.get_Y_df(RUN_1_PATH['y']).shape[0]
test_cols = dp.get_Y_df(RUN_1_PATH['y']).shape[1]
test_crop_size = PREP_image_size
test_step = PREP_crop_step

print('||||||||||||||||||||||||||||||||||||df_0||||||||||||||||||||||||||||||')
df_0 = pd.DataFrame(np.zeros((test_rows, test_cols)))
print(f'df_0.shape: ', df_0.shape)

print('||||||||||||||||||||||||||||||||||||df_2||||||||||||||||||||||||||||||')
df_2 = extend_df_for_prediction_2(df_0, test_crop_size, test_step)
print(f'df_2.shape: ', df_2.shape)

In [ ]:
test_res_0 = test_reshape_2D_Y_numpy_to_2D_0(df_0.shape[0], df_0.shape[1], 
                                        test_crop_size, test_step)

test_res_2 = test_reshape_2D_Y_numpy_to_2D_2(test_rows, test_cols, 
                                        df_2.shape[0], df_2.shape[1], 
                                        test_crop_size, test_step)

In [ ]:
# Построить карту дефектов для считанного файла
# до всяких обработок
fig, axes = plt.subplots(2)

fig.set_figwidth(16)
fig.set_figheight(10)

axes[0].pcolormesh(test_res_0)
axes[0].invert_yaxis()

axes[0].set_xlabel('Номер датчика', fontsize=15) 
axes[0].set_ylabel('Номер измерения', fontsize=15) 
axes[0].set_title(f'Распределение частот анализа каждой точки для оригинального набора данных', fontsize=15) 

#  Устанавливаем интервал основных делений: 
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(20)) 
axes[0].yaxis.set_major_locator(ticker.MultipleLocator(15)) 
 
#  Устанавливаем форматирование чисел основных делений: 
axes[0].xaxis.set_major_formatter(ticker.FormatStrFormatter('%.d')) 
axes[0].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.d')) 
 
#  Устанавливаем форматирование делений: 
axes[0].xaxis.set_tick_params(which = 'major', labelsize = 15, labelrotation = 45) 
axes[0].yaxis.set_tick_params(which = 'major', labelsize = 15) 

##############################################################################################

axes[1].pcolormesh(test_res_2)
axes[1].invert_yaxis()

axes[1].set_xlabel('Номер датчика', fontsize=15) 
axes[1].set_ylabel('Номер измерения', fontsize=15) 
axes[1].set_title(f'Распределение частот анализа каждой точки для расширенного набора данных', fontsize=15) 

#  Устанавливаем интервал основных делений: 
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(20)) 
axes[1].yaxis.set_major_locator(ticker.MultipleLocator(15)) 
 
#  Устанавливаем форматирование чисел основных делений: 
axes[1].xaxis.set_major_formatter(ticker.FormatStrFormatter('%.d')) 
axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.d')) 
 
#  Устанавливаем форматирование делений: 
axes[1].xaxis.set_tick_params(which = 'major', labelsize = 15, labelrotation = 45) 
axes[1].yaxis.set_tick_params(which = 'major', labelsize = 15) 

plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0.1, hspace=0.3)

plt.show()

# Загрузка данных

In [ ]:
# загрузка данных из файлов
X_dict = dict()
Y_dict = dict()

X_dict['df'] =  dp.get_array_like_X_df(RUN_2_PATH['x'])
Y_dict['df'] =  dp.get_Y_df(RUN_2_PATH['y'])

In [ ]:
# Запишем размеры датафреймов до обработки
ORIG_df_cols = Y_dict['df'].shape[1]
ORIG_df_rows = Y_dict['df'].shape[0]

### Оригинальная развернутая карта дефектов

In [ ]:
fig, ax = plt.subplots()

fig.set_figwidth(18)
fig.set_figheight(8)

ax.set_xlabel('Номер датчика', fontsize=25)
ax.set_ylabel('Номер измерения', fontsize=25)
ax.set_title(f"Развернутая карта дефектов", 
             fontsize=25, pad=15)

ax.patch.set_alpha(0)

#  Устанавливаем интервал основных делений:
ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
ax.yaxis.set_major_locator(ticker.MultipleLocator(20))

#  Устанавливаем форматирование чисел основных делений:
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.d'))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.d'))

#  Устанавливаем форматирование делений:
ax.tick_params(axis='both', which='both', labelsize = 25)

ax.pcolormesh(Y_dict['df'])

plt.show()

# Обработка данных

In [ ]:
# Добавление строк в датафреймы
# Справа к каждому датафрейму дописывается по 64
# элемента, чтобы сымитировать сканирование 
# трубы по всей окружности фильтром размера 64 на 64
# а еще дописывается некоторое количество строк и столбцов
# меньшее чем шаг кропа. Чтобы датафрейм можно было поделить
# на целое кол-во кропов

print('||||||||||| X df preprocessing |||||||||||')
#X_dict['df'] = dp.reshape_df_for_future_crops(X_dict['df'], PREP_image_size, PREP_crop_step)
X_dict['df'] = extend_df_for_prediction_2(X_dict['df'], PREP_image_size, PREP_crop_step)

print('||||||||||| Y df preprocessing |||||||||||')
#Y_dict['df'] = dp.reshape_df_for_future_crops(Y_dict['df'], PREP_image_size, PREP_crop_step)
Y_dict['df'] = extend_df_for_prediction_2(Y_dict['df'], PREP_image_size, PREP_crop_step)

### Константы для обработки данных

In [ ]:
# Запишем размеры датафреймов после обработки
PREP_df_cols = Y_dict['df'].shape[1]
PREP_df_rows = Y_dict['df'].shape[0]

### Расширенная развернутая карта дефектов

In [ ]:
fig, ax = plt.subplots()

fig.set_figwidth(18)
fig.set_figheight(8)

ax.set_xlabel('Номер датчика', fontsize=25)
ax.set_ylabel('Номер измерения', fontsize=25)
ax.set_title(f"Расширенная развернутая карта дефектов", 
             fontsize=25, pad=15)

ax.patch.set_alpha(0)

#  Устанавливаем интервал основных делений:
ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
ax.yaxis.set_major_locator(ticker.MultipleLocator(20))

#  Устанавливаем форматирование чисел основных делений:
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.d'))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.d'))

#  Устанавливаем форматирование делений:
ax.tick_params(axis='both', which='both', labelsize = 25)

ax.pcolormesh(Y_dict['df'])

plt.show()

# CNN с 2 входами по времени и амплитуде

### Функции для дешифровки результатов

In [ ]:
# Преобразовать массив результатов работы модели размера (batch,1)
# в матрицу размера PREP_df_rows * PREP_df_cols (карту дефектов, построенную моделью)
# Размер выходной матрицы - PREP_df_rows * PREP_df_cols
# Так как начальное расширение датафрейма на PREP_image_size по 2 осям
# тут учитывается грубо, то размер выходной матрицы будет также
# PREP_df_rows * PREP_df_cols, а ячейки по конрутру будут размыты 
# и не точны, так как по ним фильтр прошел меньше раз чем по ячейкам 
# посреди матрицы

# y = x + k
def draw_prediction_map_2_0(model, df, orig_rows_count, orig_cols_count,
                                    result_rows_count, result_cols_count, 
                                    crop_size, step = -1):

    print('||||||||||||||||||')
    print('draw_prediction_map_2_0')
    print('Crop windows height/width: ', crop_size)
    print('Crop windows step across rows and cols: ', step)

    if step == -1:
        step = crop_size
    
    new_arr = np.ones((result_rows_count, result_cols_count))     
    
    for i in range(0, result_rows_count - crop_size + 1, step): 
        print(f'row: {i} of {result_rows_count}')
        for j in range(0, result_cols_count - crop_size + 1, step):
            temp_crop = dp.pandas_crop_to_image_like_numpy(df.iloc[i:i+crop_size,j:j+crop_size])
            temp_crop = np.expand_dims(temp_crop, axis=0)
            
            res = model.predict([temp_crop[:,:,:,:32], temp_crop[:,:,:,32:]])
                   
            if res[0] > 0.5:
                new_arr[i:i+crop_size,j:j+crop_size] += res[0]
            if res[0] <= 0.5:
                new_arr[i:i+crop_size,j:j+crop_size] -= res[0]
    

    left_up_border = crop_size - 1
    right_border = result_cols_count - (result_cols_count - orig_cols_count) + 1
    bottom_border = result_rows_count - (result_rows_count - orig_rows_count) + 1
    
    print('New numpy shape: ', new_arr.shape)
    print('||||||||||||||||||\n')
    
    return new_arr, new_arr[left_up_border:,left_up_border:][:orig_rows_count,:orig_cols_count]

### Тестирование модели

In [8]:
# Загрузка модели
#loaded_model = keras.models.load_model('data//NetWork_32x32_32x32_to_1__test_0dot0_ver_10.h5')

orig = keras.models.load_model('data//NetWork_(64x32+64x32)_to(1)_(test_0dot0366)_ver_10.h5')
orig.trainable=False

#orig_inputs = [encoder.get_layer('conv_1_6').output, encoder.get_layer('conv_2_6').output]

new_input_time = Input((32,32,32), name = 'new_input_time')
input_time = UpSampling2D(2, interpolation='bilinear', name='input_time')(new_input_time)

dconv_1_1 = orig.get_layer('dconv_1_1')(input_time)
dconv_1_2 = orig.get_layer('dconv_1_2')(input_time)
dconv_1_3 = orig.get_layer('dconv_1_3')(input_time)
dconv_1_4 = orig.get_layer('dconv_1_4')(input_time)

new_input_amp = Input((32,32,32), name = 'new_input_amp')
input_amp = UpSampling2D(2, interpolation='bilinear', name='input_amp')(new_input_amp)

dconv_2_1 = orig.get_layer('dconv_2_1')(input_amp)
dconv_2_2 = orig.get_layer('dconv_2_2')(input_amp)
dconv_2_3 = orig.get_layer('dconv_2_3')(input_amp)
dconv_2_4 = orig.get_layer('dconv_2_4')(input_amp)

orig = keras.models.Model([new_input_time,new_input_amp], orig.output)
orig.summary()

TypeError: 'KerasTensor' object is not callable

In [ ]:
map_0, map_2 = draw_prediction_map_2_0(loaded_model, X_dict['df'],
                                   ORIG_df_rows, ORIG_df_cols,
                                   PREP_df_rows, PREP_df_cols, 
                                   PREP_image_size, PREP_crop_step)

# Построение карт дефектов моделью

## Грубое построение карты

In [ ]:
with plt.style.context('dark_background'):
            fig, ax = plt.subplots()
        
            fig.set_figwidth(18)
            fig.set_figheight(8)
            fig.patch.set_alpha(0.0)
        
            ax.invert_yaxis()
            ax.xaxis.tick_top()
            ax.xaxis.set_label_position('top')
            ax.set_title("Предсказанная развернутая карта дефектов", fontsize=25)
            ax.set_xlabel('Номер датчика', fontsize=20)
            ax.set_ylabel('Номер измерения', fontsize=20)
            ax.tick_params(axis='both', labelsize = 20)

            map = ax.pcolormesh(map_0)
            cbar = fig.colorbar(map)
            cbar.ax.tick_params(labelsize=20)

            ax.xaxis.set_major_locator(ticker.MultipleLocator(60))
    
plt.show()

## Умное построение карты

In [ ]:
with plt.style.context('dark_background'):
            fig, ax = plt.subplots()
        
            fig.set_figwidth(18)
            fig.set_figheight(8)
            fig.patch.set_alpha(0.0)
        
            ax.invert_yaxis()
            ax.xaxis.tick_top()
            ax.xaxis.set_label_position('top')
            ax.set_title("Предсказанная развернутая карта дефектов", fontsize=25)
            ax.set_xlabel('Номер датчика', fontsize=20)
            ax.set_ylabel('Номер измерения', fontsize=20)
            ax.tick_params(axis='both', labelsize = 20)

            map = ax.pcolormesh(map_2)
            cbar = fig.colorbar(map)
            cbar.ax.tick_params(labelsize=20)

            ax.xaxis.set_major_locator(ticker.MultipleLocator(60))
    
plt.show()